# Video Face Swap - 1 người (inswapper + InsightFace)

**Logic:** 1 video mẫu (một người chuyển động) + 1 ảnh khuôn mặt → video mới giữ nguyên
chuyển động/nền gốc, chỉ thay khuôn mặt.

**Notebook này viết lại từ bản 2 người (`video_face_swap_kiss.ipynb`)**, giữ nguyên toàn bộ
phần vá môi trường đã debug kỹ ở đó, nhưng **bỏ hết logic 2 người**:

| Đã bỏ | Vì sao chỉ cần khi có 2 người |
|---|---|
| `FaceTracker` (~150 dòng) | Khớp danh tính ArcFace + giải bài toán gán tối ưu chỉ để biết mặt nào là A, mặt nào là B. Một người thì không có gì để nhầm. |
| `swap_all_faces` + parsenet + ellipse mask (~170 dòng) | Toàn bộ chỉ để hai vùng dán không đè lên nhau lúc hôn. Một người thì `paste_back=True` mặc định của inswapper là đủ. |
| `combine_masks`, `PROTRUDE_LABELS`, xử lý che khuất đan xen | Trọng tài tranh chấp pixel giữa hai mặt. Không còn tranh chấp. |
| Cả mục 6.2 + 6.2b chẩn đoán (~11k ký tự) | Đo `margin` tracking A/B và % ô paste-back tràn sang mặt người kia. Không còn "người kia". |
| Upload 2 ảnh, quy ước A=trái / B=phải | Còn 1 ảnh. |

**Còn giữ (những thứ đã sửa đau đớn ở bản kia, vẫn cần):**
- Ghim `setuptools<82`, patch `basicsr/setup.py` cho Python 3.13, patch `torchvision.functional_tensor`
- Fallback `onnxruntime-gpu` → CPU, `ctx_id` khớp với provider
- Kiểm tra dung lượng file model sau khi `wget` (wget coi 404 là thành công)
- Lấy kích thước từ frame thật (video quay dọc có metadata rotation)
- Ghi frame thẳng vào `ffmpeg` qua pipe, mux audio trong cùng 1 pass

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

⚠️ Lưu ý: công nghệ face-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà
không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc
gắn watermark/disclosure khi xuất bản sản phẩm thật.

## 0. Cấu hình — bật/tắt làm nét mặt

**Đây là công tắc duy nhất cần chỉnh.** `USE_GFPGAN` quyết định có chạy bước làm nét / tái tạo
khuôn mặt (GFPGAN) sau khi swap hay không, và nó **tắt luôn cả phần cài đặt lẫn tải model** chứ
không chỉ tắt lúc chạy.

| | `USE_GFPGAN = True` | `USE_GFPGAN = False` |
|---|---|---|
| Cài thêm | `gfpgan`, `facexlib`, `basicsr` (phải vá source) | không |
| Tải thêm | `GFPGANv1.4.pth` (~340 MB) | không |
| Mỗi frame | detect → swap → **làm nét crop quanh mặt** | detect → swap |
| Mục 5 | chạy 3 cell vá + nạp model | in một dòng rồi bỏ qua hết |
| Kết quả | mặt nét hơn, da/tóc chi tiết hơn | mặt giữ nguyên chất lượng thô của inswapper (128×128 warp ngược) |

**Về tốc độ:** GFPGAN là một mạng riêng chạy thêm **một lượt inference nữa mỗi frame**, nên
chậm hơn là đúng — cảm nhận của bạn không sai. Ở đây nó đã được giới hạn chỉ chạy trên vùng
crop quanh mặt (không phải cả frame) nên rẻ hơn cách gọi thông thường nhiều, nhưng vẫn là chi
phí cộng thêm. **Cell cuối mục 6 in ra thời gian trung bình từng bước** (detect / swap / làm
nét) để bạn biết chính xác nó chiếm bao nhiêu phần trăm, thay vì phải đoán.

**Tắt còn một lợi ích nữa:** `basicsr` là nguồn lỗi cài đặt lớn nhất của notebook này (bug PEP
667 ở Python 3.13, bug `torchvision.functional_tensor`). `USE_GFPGAN = False` thì không đụng
tới nó, pipeline gọn và chắc hơn hẳn.

In [ ]:
# ===================== CÔNG TẮC CHÍNH =====================
USE_GFPGAN = True    # True  = swap xong làm nét mặt (đẹp hơn, chậm hơn)
                     # False = chỉ swap, không làm nét (nhanh hơn, ít lỗi cài đặt hơn)
# ==========================================================

print('USE_GFPGAN =', USE_GFPGAN)
if USE_GFPGAN:
    print('-> Sẽ cài gfpgan/facexlib/basicsr, tải thêm ~340MB weights, và làm nét mặt mỗi frame.')
else:
    print('-> Bỏ qua toàn bộ phần làm nét: không cài, không tải, không chạy. Chỉ swap.')

## 1. Cài đặt thư viện

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.
# (insightface cũng khai báo opencv-python là dependency nên chắc chắn có cv2.)

# gfpgan/facexlib chỉ cần khi làm nét. Tắt thì bỏ hẳn -> nhanh hơn và bớt một nguồn lỗi.
EXTRA_PKGS = 'gfpgan facexlib' if USE_GFPGAN else ''
print('Cài thêm:', EXTRA_PKGS or '(không có, USE_GFPGAN = False)')
!pip install -q onnxruntime-gpu {EXTRA_PKGS}

!apt-get -qq install -y ffmpeg > /dev/null
print('Xong.')

### Fix riêng cho `basicsr` (chỉ chạy khi `USE_GFPGAN = True`)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy
version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy
(thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng
chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

`basicsr` chỉ là dependency của GFPGAN, nên `USE_GFPGAN = False` thì cell này không làm gì cả.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json


def install_basicsr_patched():
    """Tải basicsr từ PyPI, vá bug PEP 667 trong setup.py, rồi cài từ source đã sửa."""
    os.makedirs('/tmp/basicsr_src', exist_ok=True)

    # Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
    # setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
    with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
        pkg_info = json.load(resp)

    sdist_url = None
    for url_info in pkg_info['urls']:
        if url_info['packagetype'] == 'sdist':
            sdist_url = url_info['url']
            break
    assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

    tar_name = sdist_url.split('/')[-1]
    tar_path = f'/tmp/basicsr_src/{tar_name}'
    urllib.request.urlretrieve(sdist_url, tar_path)
    print(f'Đã tải: {tar_name}')

    extract_dir = '/tmp/basicsr_build'
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace(...),
        # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
        root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
        assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
        root_name = root_names.pop()
        # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
        tar.extractall(extract_dir, filter='data')

    pkg_dir = os.path.join(extract_dir, root_name)
    setup_py_path = os.path.join(pkg_dir, 'setup.py')
    assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

    with open(setup_py_path, 'r') as f:
        content = f.read()

    # Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
    content = content.replace(
        "exec(compile(f.read(), version_file, 'exec'))",
        "exec(compile(f.read(), version_file, 'exec'), globals())"
    )
    content = content.replace(
        "return locals()['__version__']",
        "return globals()['__version__']"
    )

    with open(setup_py_path, 'w') as f:
        f.write(content)

    print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
    # sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
    # thay vì lệnh `pip` bất kỳ đứng đầu PATH.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
        capture_output=True, text=True
    )
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
    print('Cài basicsr thành công.')


if USE_GFPGAN:
    install_basicsr_patched()
else:
    print('USE_GFPGAN = False -> bỏ qua basicsr (chỉ là dependency của GFPGAN).')

## 2. Tải model

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính
sách, nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng
(huggingface). Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ
công rồi upload vào `/content/models/`.

Weights GFPGAN (~340 MB) **chỉ tải khi `USE_GFPGAN = True`**.

In [ ]:
import os, subprocess
os.makedirs('/content/models', exist_ok=True)

INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
GFPGAN_PATH = '/content/models/GFPGANv1.4.pth'

INSWAPPER_URL = 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'
GFPGAN_URL = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'


# Dùng subprocess thay vì `!wget`: lệnh `!` nằm trong khối `if` phụ thuộc vào chi tiết
# transform của IPython, còn subprocess thì chạy giống nhau ở mọi môi trường và
# trả về returncode để kiểm tra.
def download(url, path, min_mb, hint):
    """Tải file rồi KIỂM TRA DUNG LƯỢNG.

    wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra.
    Nếu không, mãi tới cell load model mới nổ với lỗi onnx/torch rất khó đoán nguyên nhân.
    """
    print(f'Đang tải {os.path.basename(path)} ...')
    subprocess.run(['wget', '-q', '-O', path, url])
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK  {os.path.basename(path)}: {size_mb:.1f} MB')


download(INSWAPPER_URL, INSWAPPER_PATH, 200,
         'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')

if USE_GFPGAN:
    download(GFPGAN_URL, GFPGAN_PATH, 300,
             'Kiểm tra lại link GitHub release của GFPGAN.')
else:
    print('Bỏ qua GFPGANv1.4.pth (~340MB) vì USE_GFPGAN = False.')

!ls -lh /content/models/

## 3. Upload ảnh khuôn mặt + video mẫu

Chỉ còn **1 ảnh + 1 video**. Không cần quy ước ai là A ai là B, không cần lo upload nhầm thứ tự.

In [ ]:
from google.colab import files

def pick_one(uploaded, what):
    names = list(uploaded.keys())
    assert len(names) > 0, f'Chưa upload {what} (bấm Cancel?). Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]

print('>> Upload ảnh khuôn mặt (rõ mặt, chính diện càng tốt):')
source_face_path = pick_one(files.upload(), 'ảnh mặt')

print()
print('>> Upload video mẫu:')
source_video_path = pick_one(files.upload(), 'video mẫu')

print()
print(f'Ảnh mặt : {source_face_path}')
print(f'Video   : {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại
`onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13),
sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn).

In [ ]:
import subprocess, sys, importlib

def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode

if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu cùng chiếm package `onnxruntime`. Cài cái này đè cái kia
    # là trạng thái hỏng đã biết (mất CUDAExecutionProvider, import lỗi loạn)
    # -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu==1.20.0')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Dùng providers:', providers, '| ctx_id =', ctx_id)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

swapper = insightface.model_zoo.get_model(INSWAPPER_PATH, download=False, providers=providers)

# cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
# Phải chặn ngay, không thì app.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
source_img = cv2.imread(source_face_path)
assert source_img is not None, (
    f'Không đọc được ảnh {source_face_path}. Lưu lại thành .jpg/.png rồi upload lại.'
)

faces = app.get(source_img)
assert len(faces) > 0, 'Không tìm thấy khuôn mặt trong ảnh nguồn, thử ảnh khác rõ mặt hơn.'
if len(faces) > 1:
    # Thứ tự app.get() trả về KHÔNG xác định -> không được lấy faces[0].
    # Ảnh nguồn có nhiều mặt thì lấy mặt to nhất (chủ thể của ảnh).
    print(f'  (ảnh nguồn có {len(faces)} mặt, dùng mặt lớn nhất)')
source_face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

print('Đã detect khuôn mặt nguồn thành công.')

## 5. Làm nét mặt (GFPGAN) — cả mục này tùy thuộc `USE_GFPGAN`

Nếu bạn đặt `USE_GFPGAN = False` ở mục 0 thì **cứ chạy tuần tự cả 3 cell dưới**, chúng sẽ tự
in một dòng rồi bỏ qua. Không cần nhớ bỏ cell nào.

### Fix riêng cho `basicsr` (bug với `torchvision` mới trên Colab)

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi
`basicsr` vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay
nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ
không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi
file `.py` còn tham chiếu module cũ (trong cả `basicsr`, `facexlib`, `gfpgan`).

Nó cũng tự **xoá các module hỏng khỏi `sys.modules`** cả trước lẫn sau khi vá, nhờ vậy
**không cần Runtime > Restart session**.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó.

    find_spec() chỉ định vị package, không chạy __init__.py -> không dính đúng cái
    ModuleNotFoundError mà ta đang muốn vá.
    """
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


def patch_torchvision_refs():
    purge_modules()

    targets = {}
    for name in PKGS:
        d = package_dir(name)
        if d is None:
            print(f'{name:9s}: CHƯA CÀI')
        else:
            print(f'{name:9s}: {d}')
            targets[name] = d

    assert 'basicsr' in targets, (
        'Không tìm thấy basicsr. Chạy lại cell cài basicsr ở mục 1 rồi chạy lại cell này.'
    )

    patched = []
    for name, d in targets.items():
        for p in d.rglob('*.py'):
            try:
                text = p.read_text(encoding='utf-8')
            except (UnicodeDecodeError, OSError):
                continue
            if OLD_MOD not in text:
                continue
            p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
            patched.append(p)

    print()
    if patched:
        for p in patched:
            print(f'đã vá: {p}')
    else:
        print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn functional_tensor).')

    # Purge lần nữa sau khi vá, để lần import sau đọc lại file mới trên đĩa.
    purge_modules()

    # Kiểm chứng ngay tại đây thay vì để tới cell import gfpgan mới biết.
    try:
        import basicsr.data.degradations
        print()
        print('OK: import basicsr.data.degradations thành công.')
    except Exception as e:
        print()
        print(f'VẪN LỖI: {type(e).__name__}: {e}')
        print('Nếu lỗi vẫn liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại.')
        raise


if USE_GFPGAN:
    patch_torchvision_refs()
else:
    print('USE_GFPGAN = False -> bỏ qua (không có basicsr để vá).')

In [ ]:
import subprocess, sys, importlib


def try_import_gfpgan():
    importlib.invalidate_caches()
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception: gfpgan hỏng vì basicsr/torchvision thường ném ModuleNotFoundError,
        # nhưng tuỳ phiên bản torch cũng có thể là AttributeError/OSError.
        print(f'Chưa import được gfpgan: {type(e).__name__}: {e}')
        return False


GFPGAN_AVAILABLE = False

if not USE_GFPGAN:
    print('USE_GFPGAN = False -> bỏ qua kiểm tra gfpgan.')
else:
    GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print('Thử cài lại gfpgan với log đầy đủ...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'gfpgan'],
                           capture_output=True, text=True)
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print()
        print('gfpgan không cài được -> sẽ tự động chạy tiếp mà KHÔNG làm nét.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

In [ ]:
# restorer = None nghĩa là "không làm nét". Vòng lặp ở mục 6 chỉ nhìn biến này,
# nên không cần kiểm tra USE_GFPGAN lần nữa ở trong đó.
restorer = None

if USE_GFPGAN and GFPGAN_AVAILABLE:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path=GFPGAN_PATH,
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('GFPGAN sẵn sàng (chỉ chạy trên vùng crop quanh mặt đã swap).')
elif USE_GFPGAN:
    print('USE_GFPGAN = True nhưng gfpgan không dùng được -> chạy tiếp, chỉ swap không làm nét.')
else:
    print('USE_GFPGAN = False -> chỉ swap, không làm nét.')

## 6. Xử lý video

**Đây là chỗ đơn giản đi nhiều nhất so với bản 2 người.** Mỗi frame chỉ còn ba việc:

1. `app.get(frame)` → detect mặt
2. `pick_face(...)` → chọn **mặt lớn nhất** (bản 2 người phải chạy `FaceTracker` khớp embedding
   ArcFace + giải bài toán gán tối ưu toàn cục, chỉ để phân biệt A với B)
3. `swapper.get(..., paste_back=True)` → swap và dán về luôn

Bước làm nét (4) chỉ chạy khi `restorer is not None`, tức là khi `USE_GFPGAN = True` **và**
gfpgan nạp được.

Mask dán về dùng thẳng mặc định của inswapper. Bản 2 người phải tự dựng ellipse + parsenet +
trọng tài che khuất vì ô paste-back của inswapper rộng gấp ~1.6 lần bề ngang mặt nên lúc hôn
sẽ trùm sang mặt người kia. **Một người thì không có mặt nào để trùm lên** — mask mặc định
dán vào nền, hoàn toàn vô hại.

> **Vì sao chọn mặt lớn nhất chứ không phải `faces[0]`?** Thứ tự `app.get()` trả về không
> xác định. Video có người đi ngang trong nền thì `faces[0]` có thể nhảy sang người đó ở vài
> frame rồi nhảy về — swap nhấp nháy. Mặt lớn nhất là chủ thể, ổn định giữa các frame.

Cell in ra **thời gian trung bình từng bước** ở cuối, để so sánh bật/tắt làm nét bằng số đo
thật trên đúng video của bạn.

In [ ]:
import os, sys, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

MIN_DET_SCORE = 0.5      # bỏ qua detect yếu (thường là false positive trong nền)
GFPGAN_PAD    = 0.4      # nới bbox bao nhiêu lần khi crop để làm nét
final_output  = '/content/output_final.mp4'

# globals().get: cho phép bỏ qua HẲN mục 5 (không chạy cell nào ở đó) mà vẫn chạy được
# pipeline chính, thay vì NameError: restorer.
restorer = globals().get('restorer', None)
print('Làm nét mặt:', 'BẬT' if restorer is not None else 'TẮT')


def pick_face(faces):
    """Chọn khuôn mặt để swap trong một frame. Trả về None nếu không có mặt nào đủ tốt.

    Video 1 người nên không cần tracking theo danh tính: mặt LỚN NHẤT là chủ thể.
    KHÔNG dùng faces[0] vì thứ tự app.get() trả về không xác định.
    """
    cands = [f for f in faces if f.det_score >= MIN_DET_SCORE]
    if not cands:
        return None
    return max(cands, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def enhance_face_region(img, face, pad=GFPGAN_PAD):
    """Chỉ làm nét vùng quanh khuôn mặt vừa swap, KHÔNG chạy trên cả frame.

    Gọi restorer.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với app.get() vừa chạy), làm nét luôn cả những mặt
    trong nền không hề bị swap, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = restorer.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps <= 0:       # 0.0 hoặc NaN với một số container
    print('Không đọc được fps từ video, mặc định 25.')
    fps = 25.0

# CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho progress bar.
# Vòng lặp đọc tới khi hết frame thật sự, thay vì range(total_frames) (đếm thiếu = cụt đuôi video).
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

# Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có metadata
# rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg nhận rawvideo sai size.
ret, frame = cap.read()
assert ret, 'Không đọc được frame nào từ video.'
height, width = frame.shape[:2]

print(f'Video: {width}x{height} @ {fps:.2f}fps, ~{total_frames} frames')

# Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
ffmpeg_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}', '-r', f'{fps}', '-i', 'pipe:0',
    '-i', source_video_path,
    '-map', '0:v:0', '-map', '1:a:0?',        # '?' = không có audio thì bỏ qua, không lỗi
    '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
    '-pix_fmt', 'yuv420p',                    # để trình duyệt/IPython.display.Video phát được
    '-c:a', 'aac', '-shortest',
    final_output,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

pbar = tqdm(total=total_frames, unit='frame')
frames_written = 0
frames_swapped = 0
t_detect = t_swap = t_enhance = 0.0
t_start = time.perf_counter()
rc = None
try:
    while frame is not None:
        result_frame = frame

        t0 = time.perf_counter()
        target_face = pick_face(app.get(frame))
        t1 = time.perf_counter()
        t_detect += t1 - t0

        if target_face is not None:
            result_frame = swapper.get(frame, target_face, source_face, paste_back=True)
            t2 = time.perf_counter()
            t_swap += t2 - t1

            if restorer is not None:
                result_frame = enhance_face_region(result_frame, target_face)
                t_enhance += time.perf_counter() - t2

            frames_swapped += 1

        proc.stdin.write(np.ascontiguousarray(result_frame).tobytes())
        frames_written += 1
        pbar.update(1)

        ret, frame = cap.read()
        if not ret:
            frame = None
finally:
    # Không release trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và file mp4 hỏng.
    pbar.close()
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    rc = proc.wait()

wall = time.perf_counter() - t_start
assert rc == 0, f'ffmpeg thất bại (exit code {rc}) — xem log lỗi ở trên.'

print()
print(f'Xong: {frames_written} frame, trong đó {frames_swapped} frame có mặt để swap '
      f'({frames_swapped / max(frames_written, 1):.0%}).')
print(f'Output: {final_output}')

# ---- Thời gian từng bước: để so sánh bật/tắt làm nét bằng số đo thật ----
n = max(frames_written, 1)
ns = max(frames_swapped, 1)
print()
print('Thời gian trung bình mỗi frame:')
print(f'  detect mặt : {t_detect / n * 1000:7.1f} ms')
print(f'  swap       : {t_swap / ns * 1000:7.1f} ms  (tính trên frame có mặt)')
if restorer is not None:
    print(f'  làm nét    : {t_enhance / ns * 1000:7.1f} ms  (tính trên frame có mặt)')
    busy = t_detect + t_swap + t_enhance
    print()
    print(f'-> Làm nét chiếm {t_enhance / max(busy, 1e-9):.0%} thời gian xử lý.')
    print(f'   Đặt USE_GFPGAN = False ở mục 0 rồi chạy lại để bỏ phần này.')
else:
    print('  làm nét    :     TẮT')
print()
print(f'Tổng: {wall:.1f}s cho {frames_written} frame ({wall / n * 1000:.0f} ms/frame, '
      f'{n / max(wall, 1e-9):.1f} fps xử lý).')

## 7. Kiểm tra kết quả

Audio đã được ghép ngay trong cell trên (ffmpeg nhận frame qua pipe và mux luôn audio gốc
trong cùng một pass), nên ở đây chỉ cần xác nhận file xuất ra hợp lệ.

In [ ]:
import os

assert os.path.exists(final_output) and os.path.getsize(final_output) > 0, \
    'Không tạo được video output — chạy lại cell xử lý video ở mục 6.'
print(f'{final_output}  —  {os.path.getsize(final_output) / 1e6:.1f} MB')
print()

# Xác nhận có stream video (và audio, nếu video gốc có audio)
!ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 {final_output}

## 8. Xem kết quả

In [ ]:
import os
from IPython.display import Video, display

size_mb = os.path.getsize(final_output) / 1e6
if size_mb > 50:
    # embed=True nhét toàn bộ file dưới dạng base64 vào output của notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài. Trường hợp đó thì tải về xem thay vì preview inline.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về máy.')
else:
    display(Video(final_output, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Hướng cải thiện tiếp theo

- **Bật/tắt làm nét**: đổi `USE_GFPGAN` ở **mục 0** rồi chạy lại từ đầu. Đổi giữa chừng cũng
  được, nhưng nếu chuyển từ `False` sang `True` thì phải chạy lại mục 1 (cài `basicsr`), mục 2
  (tải weights) và mục 5 — vì lần chạy trước đã bỏ qua chúng.
- **Chậm tới mức nào là bình thường?** Xem bảng thời gian ở cuối mục 6. Trên T4, `enhance` một
  crop mặt thường tốn cỡ ngang hoặc hơn `swap`, nên tắt đi có thể nhanh gần gấp đôi. Nếu muốn
  giữ làm nét mà vẫn nhanh hơn, thử giảm `GFPGAN_PAD` (crop nhỏ hơn) — nhưng đừng dưới ~0.2
  vì GFPGAN cần thấy đủ viền mặt để căn landmark.
- **Video có nhiều người mà bạn muốn thay người khác, không phải người to nhất**: sửa
  `pick_face()` ở mục 6. Ví dụ chọn theo vị trí (`min` theo `bbox[0]` = người bên trái nhất)
  thay vì theo diện tích. Nếu cần thay **hai người khác nhau** thì đây là notebook sai — dùng
  `video_face_swap_kiss.ipynb`, nó có sẵn `FaceTracker` khớp danh tính.
- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhòe nhẹ theo
  thời gian. Cải thiện bằng temporal smoothing (trung bình landmark giữa các frame liền kề)
  hoặc dùng model chuyên video như **SimSwap** chế độ video. Lưu ý GFPGAN tự nó cũng gây
  flicker vì "sáng tác" chi tiết khác nhau mỗi frame — tắt đi đôi khi lại đỡ giật hơn.
- **Frame không detect được mặt** giữ nguyên khung gốc (mặt thật lộ ra). Dòng thống kê cuối
  mục 6 cho biết tỉ lệ này — nếu thấp bất thường, thử giảm `MIN_DET_SCORE` hoặc tăng
  `det_size` ở `app.prepare()`.
- **inswapper_128 model**: link tải có thể thay đổi do các vấn đề về chính sách/gỡ bỏ. Cell
  mục 2 đã tự kiểm tra dung lượng file và báo lỗi ngay nếu tải hỏng.
- **Chất lượng ảnh mặt nguồn**: ảnh càng rõ, chính diện, ánh sáng đều thì kết quả swap càng
  tự nhiên.